<a href="https://colab.research.google.com/github/buddharajusunayana-web/Quantum/blob/main/AES_Paper_1_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import random

SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

INV_SBOX = [SBOX.index(i) for i in range(16)]

def encrypt(p, key):
    k0 = key
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

def generate_pairs(key, N):
    pairs = []
    for _ in range(N):
        p = random.randint(0, 255)
        c = encrypt(p, key)
        pairs.append((p, c))
    return pairs


def linear_attack(pairs):
    N = len(pairs)
    scores = [0] * 16

    for k in range(16):
        count = 0
        for p, c in pairs:
            v = ((c >> 4) & 0xF) ^ k
            u = INV_SBOX[v]

            lhs = (
                ((u >> 1) & 1) ^
                ((u >> 2) & 1) ^
                ((v >> 0) & 1)
            )

            if lhs == 0:
                count += 1

        scores[k] = abs(count - N/2)

    return scores


def main():
    TRUE_KEY = 0x75

    k1 = ((TRUE_KEY << 1) | (TRUE_KEY >> 7)) & 0xFF
    TRUE_SUBKEY = (k1 >> 4) & 0xF

    N = 3000

    print("Generating data...")
    pairs = generate_pairs(TRUE_KEY, N)

    print("Running linear cryptanalysis...")
    scores = linear_attack(pairs)

    recovered = max(range(16), key=lambda k: scores[k])

    prob = scores[recovered] / N + 0.5
    bias = prob - 0.5

    print("\n=========== RESULTS ===========")
    print(f"True last-round subkey : {TRUE_SUBKEY:X}")
    print(f"Recovered subkey       : {recovered:X}")
    print(f"Probability            : {prob:.4f}")
    print(f"Bias (ε)               : {bias:.4f}")
    print(f"Correct?               : {recovered == TRUE_SUBKEY}")


if __name__ == "__main__":
    main()


Generating data...
Running linear cryptanalysis...

=========== RESULTS ===========
True last-round subkey : E
Recovered subkey       : 4
Probability            : 0.6430
Bias (ε)               : 0.1430
Correct?               : False


In [ ]:
import random
import time

SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

INV_SBOX = [SBOX.index(i) for i in range(16)]

def encrypt(p, key):
    k0 = key
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

def generate_pairs(key, N):
    pairs = []
    for _ in range(N):
        p = random.randint(0, 255)
        c = encrypt(p, key)
        pairs.append((p, c))
    return pairs

def linear_attack(pairs):
    N = len(pairs)
    scores = [0] * 16

    for k in range(16):
        count = 0
        for p, c in pairs:
            v = ((c >> 4) & 0xF) ^ k
            u = INV_SBOX[v]

            lhs = (
                ((u >> 1) & 1) ^
                ((u >> 2) & 1) ^
                ((v >> 0) & 1)
            )

            if lhs == 0:
                count += 1

        scores[k] = abs(count - N / 2)

    return scores

def main():
    TRUE_KEY = 0x75

    k1 = ((TRUE_KEY << 1) | (TRUE_KEY >> 7)) & 0xFF
    TRUE_SUBKEY = (k1 >> 4) & 0xF

    N = 100000

    print("Generating data...")
    pairs = generate_pairs(TRUE_KEY, N)

    print("Running linear cryptanalysis...")
    start_time = time.time()
    scores = linear_attack(pairs)
    end_time = time.time()

    recovered = max(range(16), key=lambda k: scores[k])

    prob = scores[recovered] / N + 0.5
    bias = prob - 0.5
    exec_time = end_time - start_time

    print("\n=========== RESULTS ===========")
    print(f"True last-round subkey : {TRUE_SUBKEY:X}")
    print(f"Recovered subkey       : {recovered:X}")
    print(f"Probability            : {prob:.4f}")
    print(f"Bias (ε)               : {bias:.4f}")
    print(f"Correct?               : {recovered == TRUE_SUBKEY}")
    print(f"Execution time (s)     : {exec_time:.6f}")

if __name__ == "__main__":
    main()


Generating data...
Running linear cryptanalysis...

=========== RESULTS ===========
True last-round subkey : E
Recovered subkey       : E
Probability            : 0.6286
Bias (ε)               : 0.1286
Correct?               : True
Execution time (s)     : 0.462014


In [ ]:
import random
import time

SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

INV_SBOX = [SBOX.index(i) for i in range(16)]

def encrypt(p, key):
    k0 = key
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

def generate_differential_pairs(key, N, delta_p):
    pairs = []
    for _ in range(N):
        p1 = random.randint(0, 255)
        p2 = p1 ^ delta_p
        c1 = encrypt(p1, key)
        c2 = encrypt(p2, key)
        pairs.append((c1, c2))
    return pairs

def differential_attack(pairs, delta_u):
    scores = [0] * 16

    for k in range(16):
        count = 0
        for c1, c2 in pairs:
            v1 = ((c1 >> 4) & 0xF) ^ k
            v2 = ((c2 >> 4) & 0xF) ^ k

            u1 = INV_SBOX[v1]
            u2 = INV_SBOX[v2]

            if (u1 ^ u2) == delta_u:
                count += 1

        scores[k] = count

    return scores

def main():
    TRUE_KEY = 0x75

    k1 = ((TRUE_KEY << 1) | (TRUE_KEY >> 7)) & 0xFF
    TRUE_SUBKEY = (k1 >> 4) & 0xF

    # High-probability S-box differential
    delta_u = 0x6
    delta_p = delta_u << 4  # affect only upper nibble

    N = 100000

    print("Generating chosen plaintext pairs...")
    pairs = generate_differential_pairs(TRUE_KEY, N, delta_p)

    print("Running differential cryptanalysis...")
    start_time = time.time()
    scores = differential_attack(pairs, delta_u)
    end_time = time.time()

    recovered = max(range(16), key=lambda k: scores[k])
    exec_time = end_time - start_time

    print("\n=========== RESULTS ===========")
    print(f"True last-round subkey : {TRUE_SUBKEY:X}")
    print(f"Recovered subkey       : {recovered:X}")
    print(f"Correct?               : {recovered == TRUE_SUBKEY}")
    print(f"Execution time (s)     : {exec_time:.6f}")

    print("\nKey  Count")
    for k in range(16):
        print(f"{k:X}   {scores[k]}")

if __name__ == "__main__":
    main()


Generating chosen plaintext pairs...
Running differential cryptanalysis...

=========== RESULTS ===========
True last-round subkey : E
Recovered subkey       : E
Correct?               : True
Execution time (s)     : 0.635563

Key  Count
0   12336
1   12522
2   12517
3   12604
4   0
5   0
6   0
7   0
8   25116
9   24905
A   0
B   50021
C   24905
D   25116
E   100000
F   0


In [ ]:
!pip install pennylane-qiskit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.4/377.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 

In [ ]:
import time
import pennylane as qml
from pennylane import numpy as np

# =====================================================
# Problem Setup (Toy Cipher – Last Round Subkey Search)
# =====================================================

NUM_QUBITS = 4              # 4-bit subkey space (0..15)
TARGET_SUBKEY = 0b1110      # Correct key = hex E

dev = qml.device("default.qubit", wires=NUM_QUBITS)

# =====================================================
# Oracle: flips phase of |TARGET_SUBKEY>
# =====================================================
def oracle():
    # Convert |TARGET_SUBKEY> to |1111>
    for i in range(NUM_QUBITS):
        if ((TARGET_SUBKEY >> i) & 1) == 0:
            qml.PauliX(wires=i)

    # Phase flip using Toffoli and CNOT
    # For 4 qubits: control qubits = 0,1,2; target = 3
    # If controls all 1 → flip last qubit
    qml.Toffoli(wires=[0,1,2])
    qml.CNOT(wires=[2,3])
    qml.Toffoli(wires=[0,1,2])

    # Undo X gates
    for i in range(NUM_QUBITS):
        if ((TARGET_SUBKEY >> i) & 1) == 0:
            qml.PauliX(wires=i)

# =====================================================
# Diffusion Operator
# =====================================================
def diffusion():
    # Hadamard + X
    for i in range(NUM_QUBITS):
        qml.Hadamard(wires=i)
        qml.PauliX(wires=i)

    # Phase flip on |1111> (same trick as oracle)
    qml.Toffoli(wires=[0,1,2])
    qml.CNOT(wires=[2,3])
    qml.Toffoli(wires=[0,1,2])

    # X + Hadamard
    for i in range(NUM_QUBITS):
        qml.PauliX(wires=i)
        qml.Hadamard(wires=i)

# =====================================================
# Grover Circuit (2 iterations)
# =====================================================
@qml.qnode(dev)
def grover_circuit():
    # Initialize uniform superposition
    for i in range(NUM_QUBITS):
        qml.Hadamard(wires=i)

    # Two Grover iterations
    for _ in range(2):
        oracle()
        diffusion()

    # Return probabilities of all basis states
    return qml.probs(wires=range(NUM_QUBITS))

# =====================================================
# Run Attack
# =====================================================
def main():
    start = time.time()
    probs = grover_circuit()
    end = time.time()

    recovered = np.argmax(probs)

    print("========== QUANTUM CRYPTANALYSIS ==========")
    print(f"Recovered subkey : {hex(recovered).upper()}")
    print(f"True subkey      : {hex(TARGET_SUBKEY).upper()}")
    print(f"Success prob     : {probs[recovered]:.4f}")
    print(f"Execution time   : {end - start:.6f} seconds")

if __name__ == "__main__":
    main()


========== QUANTUM CRYPTANALYSIS ==========
Recovered subkey : 0X0
True subkey      : 0XE
Success prob     : 0.0625
Execution time   : 0.035841 seconds


In [ ]:
import pennylane as qml
from pennylane import numpy as np

NUM_QUBITS = 4
TARGET_SUBKEY = 0b1110
dev = qml.device("default.qubit", wires=NUM_QUBITS)

# -----------------------------
# Oracle: flips phase of |TARGET_SUBKEY>
# -----------------------------
def oracle():
    for i in range(NUM_QUBITS):
        if ((TARGET_SUBKEY >> i) & 1) == 0:
            qml.PauliX(wires=i)
    # Phase flip using Toffoli + CNOT (for 4 qubits)
    qml.Toffoli(wires=[0,1,2])
    qml.CNOT(wires=[2,3])
    qml.Toffoli(wires=[0,1,2])
    for i in range(NUM_QUBITS):
        if ((TARGET_SUBKEY >> i) & 1) == 0:
            qml.PauliX(wires=i)

# -----------------------------
# Diffusion Operator
# -----------------------------
def diffusion():
    for i in range(NUM_QUBITS):
        qml.Hadamard(wires=i)
        qml.PauliX(wires=i)
    # Phase flip on |1111>
    qml.Toffoli(wires=[0,1,2])
    qml.CNOT(wires=[2,3])
    qml.Toffoli(wires=[0,1,2])
    for i in range(NUM_QUBITS):
        qml.PauliX(wires=i)
        qml.Hadamard(wires=i)

# -----------------------------
# Grover Circuit
# -----------------------------
@qml.qnode(dev)
def grover_circuit():
    # Initial superposition
    for i in range(NUM_QUBITS):
        qml.Hadamard(wires=i)

    # Two Grover iterations
    for _ in range(2):
        oracle()
        diffusion()

    return qml.probs(wires=range(NUM_QUBITS))

# -----------------------------
# Draw Circuit
# -----------------------------
print(qml.draw(grover_circuit)())


0: ──H──X─╭●────╭●──X──H──X─╭●────╭●──X──H──X─╭●────╭●──X──H──X─╭●────╭●──X──H─┤ ╭Probs
1: ──H────├●────├●──H──X────├●────├●──X──H────├●────├●──H──X────├●────├●──X──H─┤ ├Probs
2: ──H────╰X─╭●─╰X──H──X────╰X─╭●─╰X──X──H────╰X─╭●─╰X──H──X────╰X─╭●─╰X──X──H─┤ ├Probs
3: ──H───────╰X──H──X──────────╰X──X──H──────────╰X──H──X──────────╰X──X──H────┤ ╰Probs


In [ ]:
import time

# ===============================
# Toy cipher definition
# ===============================

SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

INV_SBOX = [SBOX.index(i) for i in range(16)]

def encrypt(p, key):
    k0 = key
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0xF]
    return v ^ k1

# ===============================
# Integral plaintext set
# ===============================

def generate_integral_set(constant_low_nibble=0x0):
    """
    Upper nibble varies over 0..15
    Lower nibble fixed
    """
    return [(x << 4) | constant_low_nibble for x in range(16)]

# ===============================
# Integral (Square) Attack
# ===============================

def integral_attack(ciphertexts):
    scores = [0] * 16

    for k in range(16):
        xor_sum = 0
        for c in ciphertexts:
            v = ((c >> 4) & 0xF) ^ k   # guess subkey
            u = INV_SBOX[v]           # undo S-box
            xor_sum ^= u

        if xor_sum == 0:
            scores[k] = 1  # balanced

    return scores

# ===============================
# Main execution
# ===============================

def main():
    start_time = time.time()

    TRUE_KEY = 0x75

    k1 = ((TRUE_KEY << 1) | (TRUE_KEY >> 7)) & 0xFF
    TRUE_SUBKEY = (k1 >> 4) & 0xF

    print("True last-round subkey:", hex(TRUE_SUBKEY).upper())

    # Step 1: Generate integral plaintexts
    plaintexts = generate_integral_set()

    # Step 2: Encrypt
    ciphertexts = [encrypt(p, TRUE_KEY) for p in plaintexts]

    # Step 3: Run integral attack
    scores = integral_attack(ciphertexts)

    recovered = scores.index(1)

    end_time = time.time()

    print("\nKey  Balanced")
    for k in range(16):
        print(f"{k:X}    {scores[k]}")

    print("\nRecovered subkey :", hex(recovered).upper())
    print("Correct?         :", recovered == TRUE_SUBKEY)
    print(f"Execution time   : {end_time - start_time:.6f} seconds")

if __name__ == "__main__":
    main()


True last-round subkey: 0XE

Key  Balanced
0    1
1    1
2    1
3    1
4    1
5    1
6    1
7    1
8    1
9    1
A    1
B    1
C    1
D    1
E    1
F    1

Recovered subkey : 0X0
Correct?         : False
Execution time   : 0.000168 seconds


In [ ]:
import time
import random

# ===============================
# Toy Cipher Definition
# ===============================

SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

def encrypt_round1(p, k0):
    u = p ^ k0
    return (SBOX[u >> 4] << 4) | SBOX[u & 0xF]

def encrypt_round2(v, k1):
    return v ^ k1

def encrypt(p, key):
    k0 = key
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    return encrypt_round2(encrypt_round1(p, k0), k1)

# ===============================
# MITM for ONE sample
# ===============================

def mitm_single_sample(p, c):
    forward = {}
    for k0 in range(256):
        mid = encrypt_round1(p, k0)
        forward.setdefault(mid, []).append(k0)

    candidates = set()
    for k1 in range(256):
        mid = c ^ k1
        if mid in forward:
            for k0 in forward[mid]:
                candidates.add((k0, k1))

    return candidates

# ===============================
# MITM using MANY samples
# ===============================

def mitm_attack_many_samples(true_key, num_samples=100000):
    start = time.time()

    # Generate samples
    samples = []
    for _ in range(num_samples):
        p = random.randint(0, 255)
        c = encrypt(p, true_key)
        samples.append((p, c))

    # MITM on first sample
    p0, c0 = samples[0]
    candidates = mitm_single_sample(p0, c0)

    # Intersect with remaining samples
    for p, c in samples[1:]:
        new_candidates = set()
        for k0, k1 in candidates:
            if encrypt_round2(encrypt_round1(p, k0), k1) == c:
                new_candidates.add((k0, k1))
        candidates = new_candidates

        if len(candidates) == 1:
            break

    end = time.time()
    return candidates, end - start

# ===============================
# Run the Attack
# ===============================

def main():
    TRUE_KEY = 0x75
    TRUE_K1 = ((TRUE_KEY << 1) | (TRUE_KEY >> 7)) & 0xFF
    NUM_SAMPLES = 100000

    candidates, exec_time = mitm_attack_many_samples(TRUE_KEY, NUM_SAMPLES)

    print("========== MITM (100000 SAMPLES) ==========")
    print(f"True key        : k0 = 0x{TRUE_KEY:02X}, k1 = 0x{TRUE_K1:02X}")
    print(f"Samples used    : {NUM_SAMPLES}")
    print(f"Candidates left : {len(candidates)}")

    for k0, k1 in candidates:
        print(f"Recovered key   : k0 = 0x{k0:02X}, k1 = 0x{k1:02X}")

    print(f"Correct?        : {(TRUE_KEY, TRUE_K1) in candidates}")
    print(f"Execution time  : {exec_time:.6f} seconds")

if __name__ == "__main__":
    main()


========== MITM (100000 SAMPLES) ==========
True key        : k0 = 0x75, k1 = 0xEA
Samples used    : 100000
Candidates left : 1
Recovered key   : k0 = 0x75, k1 = 0xEA
Correct?        : True
Execution time  : 0.185667 seconds


In [ ]:
import time
import random

# ===============================
# Toy Cipher Definition
# ===============================

SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

def encrypt(p, key):
    k0 = key
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

# ===============================
# Brute Force Attack
# ===============================

def brute_force_attack(samples):
    start = time.time()

    for key in range(256):
        valid = True
        for p, c in samples:
            if encrypt(p, key) != c:
                valid = False
                break
        if valid:
            end = time.time()
            return key, end - start

    end = time.time()
    return None, end - start

# ===============================
# Run the Attack
# ===============================

def main():
    TRUE_KEY = 0x75
    NUM_SAMPLES = 100000

    # Generate samples
    samples = []
    for _ in range(NUM_SAMPLES):
        p = random.randint(0, 255)
        c = encrypt(p, TRUE_KEY)
        samples.append((p, c))

    recovered_key, exec_time = brute_force_attack(samples)

    print("========== BRUTE FORCE ATTACK ==========")
    print(f"Samples used   : {NUM_SAMPLES}")
    print(f"Recovered key  : 0x{recovered_key:02X}")
    print(f"True key       : 0x{TRUE_KEY:02X}")
    print(f"Correct?       : {recovered_key == TRUE_KEY}")
    print(f"Execution time : {exec_time:.6f} seconds")

if __name__ == "__main__":
    main()


========== BRUTE FORCE ATTACK ==========
Samples used   : 100000
Recovered key  : 0x75
True key       : 0x75
Correct?       : True
Execution time : 0.024982 seconds


In [ ]:
pip install pycosat


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pycosat: filename=pycosat-0.6.6-cp312-cp312-linux_x86_64.whl size=170300 sha256=fda6b2c94cdda6a8432624ef076676bd0ebc4757b65e7ab12cb8275d5d6d7c16
  Stored in directory: /root/.cache/pip/wheels/a2/34/2e/81095f4bfa4d06004e3bebc7e415733c40f0d0ab583100d3af
Successfully built pycosat


In [ ]:
"""
Algebraic-style attack (CNF + SAT) for the toy cipher.
- Tries to use pycosat if installed.
- Falls back to brute force if pycosat not available.
- Measures execution time of solver step.
"""

import time
import itertools

# -------------------------
# Toy cipher components
# -------------------------
SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]
INV_SBOX = [SBOX.index(i) for i in range(16)]

def encrypt(p, key):
    k0 = key
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

# -------------------------
# CNF helper utilities
# -------------------------
# We'll number variables as integers for CNF:
# We'll create variables for the 8 key bits (k7..k0 -> var ids)
# and for intermediate u bits per sample if needed.
#
# Variable indexing layout:
# 1..8 : key bits k0..k7 (k0 is least-significant)
# Additional vars for each sample's intermediate bits start at 9 upward.
#
# CNF clauses are lists of integers (positive for var, negative for negated var).
# Example: (x1 OR NOT x2) -> [1, -2]
#
# We'll encode S-box mapping by enumerating allowed input-output assignments
# and for each forbidden assignment add clauses (direct encoding).
#
# This is not the most compact CNF, but it's straightforward and small for toy cipher.

def bit_list(x, width):
    return [(x >> i) & 1 for i in range(width)]

def new_var(var_counter):
    var_counter[0] += 1
    return var_counter[0]

# Encode equality of a variable to a constant: x == val  -> clause [x] or [-x]
def eq_var_const(var, val):
    return [[var]] if val == 1 else [[-var]]

# Encode that a 4-bit vector (vars[]) equals a 4-bit constant value (0..15)
def eq_bits_const(vars4, val):
    b = bit_list(val, 4)
    clauses = []
    for var, bit in zip(vars4, b):
        clauses += eq_var_const(var, bit)
    return clauses

# Encode S-box relation: input 4-bit variables -> output 4-bit variables
# We use direct table encoding: for each input value x, allowed output is SBOX[x]
# We'll enforce: for some x, input==x -> output==SBOX[x]
# Which can be encoded as for each x: (NOT(in==x) OR (out==SBOX[x]))
# Expand in CNF by turning (in==x) into conjunction of literals, and (out==y) into conjunction.
def encode_sbox_relation(in_vars, out_vars, var_counter):
    # We encode as series of implications:
    # For each x in 0..15:
    #    (in==x) -> (out==SBOX[x])
    #    translates to clauses: for each bit of out that is 1, (NOT(in==x) OR out_bit)
    #    and for each bit of out that is 0, (NOT(in==x) OR NOT out_bit)
    # Also we need to ensure *some* x matches (input must be one of 0..15) — but input always is 4 bits so it's covered.
    clauses = []
    # Build conjunction literal for (in==x) as list of literals: if bit==1 -> var, else -> -var
    for x in range(16):
        in_lits = []
        for i, v in enumerate(in_vars):
            bit = (x >> i) & 1
            in_lits.append(v if bit == 1 else -v)
        out_val = SBOX[x]
        for i, v in enumerate(out_vars):
            out_bit = (out_val >> i) & 1
            # clause: (NOT(in==x) OR (out_bit literal))
            # NOT(in==x) is a disjunction of negated in_lits; to convert implication to CNF we add
            # one clause per out-bit: (neg(in_bit1) OR neg(in_bit2) OR ... OR out_lit)
            clause = [-(lit) for lit in in_lits]
            clause.append(v if out_bit == 1 else -v)
            clauses.append(clause)
    return clauses

# Encode relation v = u ^ k for 8-bit values, given lists of bit-variable indices for v and k bits (k are key bit vars)
# Here v and u will be concrete constants (cipher text bits or intermediate bits) or variables depending on context.
# We'll implement only the case we need: v = c ^ k1 where c (cipher) is known and k1 is derived from key bits.
# To keep things simple we will compute k1 bits directly as expressions of key bits and then equate v bits to known constants.
# So we don't need clause encoding for XOR; we'll compute k1 bits numerically and use eq_bits_const.

# -------------------------
# Build CNF for samples
# -------------------------
def build_cnf_for_samples(samples):
    """
    samples: list of (p, c) plaintext-ciphertext integer pairs (8-bit)
    returns: (clauses, num_vars)
    variables:
      1..8 : key bits k0..k7 (k0 least significant)
      for each sample s:
        u_s_0 .. u_s_7  intermediate bits after XOR with k0 (but before S-box) OR we can omit and compute them as expressions
    We'll introduce for each sample 4 variables u_high bits after inverse Sbox of (c ^ k1)'s upper nibble,
    because we need to equate Sbox(u_upper) to v_upper, but typical approach is:
      - Introduce variables for u_upper_bits (4 vars per sample),
      - enforce that SBOX(u_upper) equals v_upper where v_upper is ((c ^ k1) >> 4) & 0xF (depends on key bits)
    For the toy scale we can keep it simple: introduce u_upper bits variables per sample and encode Sbox relation
    between u_upper and the v_upper variables (which are tied to ciphertext and k1 computed).
    """
    clauses = []
    var_counter = [8]  # key bits 1..8 reserved

    # key bit variables: 1..8 (k0..k7)
    key_vars = list(range(1, 9))


    for (p, c) in samples:
        # For every candidate key K, see if it matches this sample
        for K in range(256):
            if encrypt(p, K) != c:
                # forbid assignment kbits == bits(K)
                clause = []
                for i in range(8):  # bit i (0 LSB)
                    bit = (K >> i) & 1
                    var = key_vars[i]
                    # to require (key_bits != K_bits) we add literal that is true when var != bit
                    # if bit == 1 -> (var != 1) -> -var; if bit == 0 -> (var != 0) -> var
                    clause.append(-var if bit == 1 else var)
                clauses.append(clause)
        # after processing sample continue
    num_vars = var_counter[0]
    return clauses, num_vars

# -------------------------
# Try to solve CNF with pycosat (if available), else brute-force fallback
# -------------------------
def solve_cnf(clauses, num_vars):
    try:
        import pycosat
    except Exception:
        pycosat = None

    start = time.time()
    if pycosat is not None:
        # pycosat solves CNF and returns a satisfying assignment or 'UNSAT'
        sol = pycosat.solve(clauses)
        end = time.time()
        if sol == "UNSAT":
            return [], end - start
        # sol is list of integers for variables that are true (positive) and negative for false
        # We'll parse all satisfying assignments by enumerating (pycosat has 'itersolve' maybe)
        # For simplicity just return this assignment as recovered key
        assignment = sol
        key = 0
        for lit in assignment:
            if abs(lit) <= 8:
                var = abs(lit)
                val = 1 if lit > 0 else 0
                if val:
                    key |= (1 << (var - 1))
        return [key], end - start
    else:
        # Fallback: brute-force search for keys satisfying all clauses (equivalent to earlier elimination)
        # We'll interpret each clause as disjunction, check if key assignment satisfies all clauses
        def key_satisfies_clauses(K):
            # build key bit assignment map
            assign = {}
            for i in range(8):
                assign[i+1] = 1 if ((K >> i) & 1) else 0
            for clause in clauses:
                satisfied = False
                for lit in clause:
                    var = abs(lit)
                    sign = 1 if lit > 0 else 0
                    if assign[var] == sign:
                        satisfied = True
                        break
                if not satisfied:
                    return False
            return True

        solutions = []
        for K in range(256):
            if key_satisfies_clauses(K):
                solutions.append(K)
        end = time.time()
        return solutions, end - start

# -------------------------
# Main runner
# -------------------------
def algebraic_attack(samples):
    # samples: list of (p,c) pairs
    clauses, num_vars = build_cnf_for_samples(samples)
    solutions, solve_time = solve_cnf(clauses, num_vars)
    return solutions, solve_time

# -------------------------
# Example usage
# -------------------------
def main():
    TRUE_KEY = 0x75
    NUM_SAMPLES = 100000  # you can adjust number of samples
    random_seed = 0xC0FFEE

    import random
    random.seed(random_seed)
    samples = []
    for _ in range(NUM_SAMPLES):
        p = random.randint(0, 255)
        c = encrypt(p, TRUE_KEY)
        samples.append((p, c))

    print(f"Running algebraic-style attack on {NUM_SAMPLES} random samples...")
    sol, t = algebraic_attack(samples)
    print(f"Solve time : {t:.6f} seconds")
    if not sol:
        print("No solution found (UNSAT).")
    else:
        print(f"Number of candidate keys found: {len(sol)}")
        for k in sol[:16]:
            print(f"0x{k:02X}", end=" ")
        print()
        found_true = TRUE_KEY in sol
        print("True key present?:", found_true)

if __name__ == "__main__":
    main()


Running algebraic-style attack on 100000 random samples...
Solve time : 21.673247 seconds
Number of candidate keys found: 1
0x75 
True key present?: True


In [ ]:
!pip install pennylane


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 31.4 MB/s eta 0:00:00


In [ ]:
import random
import math
import time

# =====================================================
# Toy Cipher
# =====================================================
SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

def encrypt(p, key):
    k0 = key & 0xFF
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

# =====================================================
# Generate Differential Samples
# =====================================================
def generate_samples(key, delta, samples):
    data = []
    for _ in range(samples):
        p = random.randint(0, 255)
        p2 = p ^ delta
        diff = encrypt(p, key) ^ encrypt(p2, key)
        data.append((p, p2, diff))
    return data

# =====================================================
# Fast Differential Filter with Early Rejection
# =====================================================
def differential_filter(samples):
    candidates = []
    for k in range(256):
        ok = True
        for p1, p2, expected in samples:
            if encrypt(p1, k) ^ encrypt(p2, k) != expected:
                ok = False
                break  # EARLY REJECTION
        if ok:
            candidates.append(k)
    return candidates

# =====================================================
# Grover Simulation
# =====================================================
def grover_simulation(marked, N=256):
    amplitudes = [1 / math.sqrt(N)] * N
    M = len(marked)

    if M == 0:
        return -1, amplitudes

    iterations = round(math.pi / 4 * math.sqrt(N / M))

    for _ in range(iterations):
        for m in marked:
            amplitudes[m] *= -1
        mean = sum(amplitudes) / N
        amplitudes = [2 * mean - a for a in amplitudes]

    probs = [a * a for a in amplitudes]
    recovered = max(range(N), key=lambda i: probs[i])
    return recovered, probs

# =====================================================
# Quantum Differential Cryptanalysis
# =====================================================
def quantum_differential_attack():
    TRUE_KEY = 0x75
    deltas = [0x0F, 0x33, 0x55]   # strong + independent
    SAMPLES_PER_DELTA = 100000     # Increased to 100k samples per delta

    print("Generating chosen plaintext pairs...")

    start = time.time()
    candidates = set(range(256))

    filter_start = time.time()
    for d in deltas:
        samples = generate_samples(TRUE_KEY, d, SAMPLES_PER_DELTA)
        valid = set(differential_filter(samples))
        candidates &= valid
    filter_end = time.time()

    recovered, probs = grover_simulation(list(candidates))
    end = time.time()

    print("\n========== QUANTUM DIFFERENTIAL CRYPTANALYSIS ==========")
    print(f"Candidates remaining : {len(candidates)}")
    print(f"Candidate keys       : {[hex(k) for k in candidates]}")
    print(f"Recovered key        : 0x{recovered:02X}")
    print(f"True key             : 0x{TRUE_KEY:02X}")
    print(f"Recovered == True?   : {recovered == TRUE_KEY}")
    print(f"Filter time          : {filter_end - filter_start:.6f} s")
    print(f"Total execution time : {end - start:.6f} s")
    print(f"Recovered probability: {probs[recovered]:.6f}")

# =====================================================
# Run
# =====================================================
if __name__ == "__main__":
    quantum_differential_attack()

Generating chosen plaintext pairs...

========== QUANTUM DIFFERENTIAL CRYPTANALYSIS ==========
Candidates remaining : 1
Candidate keys       : ['0x75']
Recovered key        : 0x75
True key             : 0x75
Recovered == True?   : True
Filter time          : 4.151438 s
Total execution time : 4.152063 s
Recovered probability: 0.986186


In [ ]:
# grover_fullkey_sim.py
# Pure-Python Grover simulation over full 8-bit keyspace (256 states).
# Oracle marks keys that are consistent with the provided samples (deterministic check).
# This recovers the full 8-bit master key robustly (no PennyLane required).

import math
import random
import time
from typing import List, Tuple

# -------------------------
# Toy cipher (same as before)
# -------------------------
SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

def encrypt(p: int, key: int) -> int:
    k0 = key & 0xFF
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

# -------------------------
# Data generation helpers
# -------------------------
def gen_plaintext_cipher_pairs(key: int, N: int) -> List[Tuple[int,int]]:
    return [(random.randint(0,255), encrypt(random.randint(0,255), key)) for _ in range(N)]
    # Note: above incorrectly encrypts random plaintext twice; better to generate properly:
def gen_random_pairs(key: int, N: int) -> List[Tuple[int,int]]:
    pairs = []
    for _ in range(N):
        p = random.randint(0,255)
        c = encrypt(p, key)
        pairs.append((p,c))
    return pairs

def gen_chosen_diff_pairs(key: int, N: int, delta_p: int) -> List[Tuple[int,int]]:
    pairs = []
    for _ in range(N):
        p1 = random.randint(0,255)
        p2 = p1 ^ delta_p
        c1 = encrypt(p1, key)
        c2 = encrypt(p2, key)
        pairs.append((c1,c2))
    return pairs

# -------------------------
# Find keys consistent with dataset (deterministic check)
# -------------------------
def find_consistent_master_keys(pairs: List[Tuple[int,int]]) -> List[int]:
    """
    For each candidate master key K (0..255), check if encrypt(p,K) == c for all (p,c) in pairs.
    Return list of keys that pass.
    """
    candidates = []
    for K in range(256):
        ok = True
        for p,c in pairs:
            if encrypt(p, K) != c:
                ok = False
                break
        if ok:
            candidates.append(K)
    return candidates

def find_consistent_master_keys_from_diff(pairs: List[Tuple[int,int]], delta_p: int) -> List[int]:
    """
    For differential pairs stored as (c1,c2), check K consistency:
    For each candidate key K, encrypt p1 and p2? We don't have p1/p2 here, so this function assumes the pairs are (c1,c2)
    produced from known p1,p2 with delta_p. For simplicity in this script we will use the classical approach:
    we will instead regenerate pairs as (p1,p2,c1,c2) when needed. To keep things simple, we will use plaintext-ciphertext pairs
    for full-key verification in both linear and differential flows (i.e., verify master key directly).
    """
    # not used in this script
    return []

# -------------------------
# Grover simulation (statevector)
# -------------------------
def grover_search_marking_set(marked_indices: List[int], iterations: int) -> Tuple[int, List[float]]:
    """
    Simulate Grover on N=256 basis states.
    - marked_indices: list of integer indices (0..255) to be phase-flipped by the oracle
    - iterations: number of Grover iterations to perform (oracle+diffusion)
    Returns: (recovered_index, probabilities list)
    """
    N = 256
    # initialize uniform amplitude vector
    amp = [1.0/math.sqrt(N) for _ in range(N)]

    for it in range(iterations):
        # Oracle: phase flip for marked indices
        for idx in marked_indices:
            amp[idx] = -amp[idx]
        # Diffusion: reflect about mean
        mean = sum(amp)/N
        amp = [2*mean - a for a in amp]

    probs = [abs(a)**2 for a in amp]
    recovered = max(range(N), key=lambda i: probs[i])
    return recovered, probs

# -------------------------
# Utility: choose iteration count (approx)
# -------------------------
def optimal_grover_iterations(N: int, t_marked: int) -> int:
    """
    Roughly optimal iterations ~ floor(pi/(4*theta)) where sin(theta) = sqrt(t_marked/N)
    If t_marked==0 -> return 0
    """
    if t_marked == 0:
        return 0
    theta = math.asin(math.sqrt(t_marked / N))
    k_opt = int(round((math.pi / (4 * theta)) - 0.5))
    if k_opt < 0:
        k_opt = 0
    return k_opt

# -------------------------
# End-to-end retrieval using Grover over full master key
# -------------------------
def grover_recover_full_key_from_pc_pairs(pairs: List[Tuple[int,int]], max_iter_override: int = None):
    """
    1. Find keys consistent with all supplied (p,c) pairs (deterministic filter).
    2. Use Grover to amplify among full 256 state space, marking the (usually single) consistent key(s).
    3. Return recovered key and diagnostics.
    """
    start_filter = time.time()
    consistent = find_consistent_master_keys(pairs)
    end_filter = time.time()

    if len(consistent) == 0:
        return {
            "status": "no_consistent_key",
            "consistent_keys": consistent,
            "filter_time": end_filter - start_filter
        }

    # mark consistent keys
    marked = consistent.copy()

    # choose number iterations
    N = 256
    t_marked = len(marked)
    if max_iter_override is not None:
        iters = max_iter_override
    else:
        iters = optimal_grover_iterations(N, t_marked)
        if iters == 0:
            iters = 1  # at least one iteration if there is a marked item

    # run Grover simulation
    start_q = time.time()
    recovered_idx, probs = grover_search_marking_set(marked, iterations=iters)
    end_q = time.time()

    return {
        "status": "ok",
        "consistent_keys": consistent,
        "marked_count": t_marked,
        "iters_used": iters,
        "recovered_index": recovered_idx,
        "recovered_key": recovered_idx,
        "quantum_time": end_q - start_q,
        "filter_time": end_filter - start_filter,
        "probs": probs
    }

# -------------------------
# Demo / run
# -------------------------
def main():
    random.seed(0)

    TRUE_KEY = 0x75
    NUM_SAMPLES = 100000   # number of plaintext-ciphertext pairs to generate and use
    print("Generating", NUM_SAMPLES, "random plaintext-ciphertext samples...")
    samples = gen_random_pairs(TRUE_KEY, NUM_SAMPLES)

    print("Filtering keys consistent with samples (classical check)...")
    start = time.time()
    consistent = find_consistent_master_keys(samples)
    filt_time = time.time() - start
    print("Filter time: {:.6f} s".format(filt_time))
    print("Number of consistent keys found:", len(consistent))
    if len(consistent) <= 10:
        print("Consistent keys:", [hex(k) for k in consistent][:50])
    else:
        print("Many consistent keys (showing first 10):", [hex(k) for k in consistent[:10]])

    # Usually consistent == [TRUE_KEY]. If so, Grover will mark single state.
    # Now run Grover simulation over full key space marking the consistent key(s)
    print("\nRunning Grover simulation (marks consistent keys)...")
    result = grover_recover_full_key_from_pc_pairs(samples)
    if result["status"] != "ok":
        print("No consistent keys found — cannot run Grover.")
        return

    recovered = result["recovered_key"]
    print("Marked count:", result["marked_count"])
    print("Grover iterations used:", result["iters_used"])
    print("Recovered key (index): 0x{:02X}".format(recovered))
    print("True key: 0x{:02X}".format(TRUE_KEY))
    print("Recovered == True?:", recovered == TRUE_KEY)
    print("Quantum (sim) time: {:.6f} s".format(result["quantum_time"]))
    print("Filter time: {:.6f} s".format(result["filter_time"]))

    # show probability of recovered state
    prob = result["probs"][recovered]
    print("Probability of recovered state after Grover:", prob)

if __name__ == "__main__":
    main()


Generating 100000 random plaintext-ciphertext samples...
Filtering keys consistent with samples (classical check)...
Filter time: 0.026635 s
Number of consistent keys found: 1
Consistent keys: ['0x75']

Running Grover simulation (marks consistent keys)...
Marked count: 1
Grover iterations used: 12
Recovered key (index): 0x75
True key: 0x75
Recovered == True?: True
Quantum (sim) time: 0.000334 s
Filter time: 0.026669 s
Probability of recovered state after Grover: 0.9999470421032738


In [ ]:
import random
import math
import time

# =====================================================
# Toy Cipher
# =====================================================
SBOX = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

def encrypt(p, key):
    k0 = key & 0xFF
    k1 = ((key << 1) | (key >> 7)) & 0xFF
    u = p ^ k0
    v = (SBOX[u >> 4] << 4) | SBOX[u & 0x0F]
    return v ^ k1

# =====================================================
# Generate Differential Samples
# =====================================================
def generate_samples(key, delta, samples):
    data = []
    for _ in range(samples):
        p = random.randint(0, 255)
        p2 = p ^ delta
        diff = encrypt(p, key) ^ encrypt(p2, key)
        data.append((p, p2, diff))
    return data

# =====================================================
# Fast Differential Filter with Early Rejection
# =====================================================
def differential_filter(samples):
    candidates = []
    for k in range(256):
        ok = True
        for p1, p2, expected in samples:
            if encrypt(p1, k) ^ encrypt(p2, k) != expected:
                ok = False
                break  # EARLY REJECTION
        if ok:
            candidates.append(k)
    return candidates

# =====================================================
# Grover Simulation
# =====================================================
def grover_simulation(marked, N=256):
    amplitudes = [1 / math.sqrt(N)] * N
    M = len(marked)

    if M == 0:
        return -1, amplitudes

    iterations = round(math.pi / 4 * math.sqrt(N / M))

    for _ in range(iterations):
        for m in marked:
            amplitudes[m] *= -1
        mean = sum(amplitudes) / N
        amplitudes = [2 * mean - a for a in amplitudes]

    probs = [a * a for a in amplitudes]
    recovered = max(range(N), key=lambda i: probs[i])
    return recovered, probs

# =====================================================
# Quantum Differential Cryptanalysis
# =====================================================
def quantum_differential_attack():
    TRUE_KEY = 0x75
    deltas = [0x0F, 0x33, 0x55]   # strong + independent
    SAMPLES_PER_DELTA = 3000     # enough to isolate key

    print("Generating chosen plaintext pairs...")

    start = time.time()
    candidates = set(range(256))

    filter_start = time.time()
    for d in deltas:
        samples = generate_samples(TRUE_KEY, d, SAMPLES_PER_DELTA)
        valid = set(differential_filter(samples))
        candidates &= valid
    filter_end = time.time()

    recovered, probs = grover_simulation(list(candidates))
    end = time.time()

    print("\n========== QUANTUM DIFFERENTIAL CRYPTANALYSIS ==========")
    print(f"Candidates remaining : {len(candidates)}")
    print(f"Candidate keys       : {[hex(k) for k in candidates]}")
    print(f"Recovered key        : 0x{recovered:02X}")
    print(f"True key             : 0x{TRUE_KEY:02X}")
    print(f"Recovered == True?   : {recovered == TRUE_KEY}")
    print(f"Filter time          : {filter_end - filter_start:.6f} s")
    print(f"Total execution time : {end - start:.6f} s")
    print(f"Recovered probability: {probs[recovered]:.6f}")

# =====================================================
# Run
# =====================================================
if __name__ == "__main__":
    quantum_differential_attack()


Generating chosen plaintext pairs...

========== QUANTUM DIFFERENTIAL CRYPTANALYSIS ==========
Candidates remaining : 1
Candidate keys       : ['0x75']
Recovered key        : 0x75
True key             : 0x75
Recovered == True?   : True
Filter time          : 0.084882 s
Total execution time : 0.085203 s
Recovered probability: 0.986186
